In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
train_data = pd.read_csv(r"C:\Users\sidhu\OneDrive\Desktop\Kaggle\Titanic - Machine Learning from Disaster\data\train.csv")
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
test_data = pd.read_csv(r"C:\Users\sidhu\OneDrive\Desktop\Kaggle\Titanic - Machine Learning from Disaster\data\train.csv")
test_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
surviving_women = train_data.loc[train_data.Sex == "female"]["Survived"]
percent_of_women_alive = sum(surviving_women)/len(surviving_women) * 100

print("% of women who survived : ",percent_of_women_alive)

% of women who survived :  74.20382165605095


In [5]:
surviving_men = train_data.loc[train_data.Sex == "male"]["Survived"]
percent_of_men_alive = sum(surviving_men)/len(surviving_men) * 100

print("% of men who survived : ", percent_of_men_alive)

% of men who survived :  18.890814558058924


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import optuna

y = train_data["Survived"]

features = ["Parch", "SibSp", "Sex", "Pclass"]

X = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])

X_test = X_test.reindex(columns=X.columns, fill_value=0)

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=1
)


def model_evaluation(trial):

    model = RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 50, 500),
        max_depth=trial.suggest_int("max_depth", 2, 30),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
        max_features=trial.suggest_categorical(
            "max_features",
            ["sqrt", "log2", None]
        ),
        random_state=1
    )

    model.fit(X_train, y_train)

    val_predictions = model.predict(X_val)

    val_accuracy = accuracy_score(y_val, val_predictions)

    return val_accuracy


study = optuna.create_study(direction="maximize")

study.optimize(model_evaluation, n_trials=100)

print("Best parameters:\n", study.best_params)

print("\nBest validation accuracy percentage:\n", study.best_value * 100, "%")

[I 2026-08-12 00:31:34,602] A new study created in memory with name: no-name-807a0065-2906-4677-9f15-158a0502e650
[I 2026-08-12 00:31:35,587] Trial 0 finished with value: 0.8268156424581006 and parameters: {'n_estimators': 382, 'max_depth': 14, 'min_samples_split': 15, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.8268156424581006.
[I 2026-08-12 00:31:36,065] Trial 1 finished with value: 0.8100558659217877 and parameters: {'n_estimators': 183, 'max_depth': 28, 'min_samples_split': 7, 'min_samples_leaf': 6, 'max_features': None}. Best is trial 0 with value: 0.8268156424581006.
[I 2026-08-12 00:31:36,996] Trial 2 finished with value: 0.8156424581005587 and parameters: {'n_estimators': 353, 'max_depth': 30, 'min_samples_split': 20, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.8268156424581006.
[I 2026-08-12 00:31:37,335] Trial 3 finished with value: 0.8044692737430168 and parameters: {'n_estimators': 144, 'max_depth': 15, 'm

Best parameters:
 {'n_estimators': 464, 'max_depth': 7, 'min_samples_split': 2, 'min_samples_leaf': 7, 'max_features': 'sqrt'}

Best validation accuracy percentage:
 83.24022346368714 %


I made use of Optuna (hyperparamter optimization framework that makes use of Bayesian style of optimization) to intelligently select ideal set of hyperparamter values after manually giving either a set of values to choose from or a range of values, within which to choose from, for each hyper parameter.

 Something i missed earlier which is a crucial step to maximise accuracy, is understanding the inter-relationships within the feature set via EDA and doing feature engineering to select the best set of features. This will be covered in the next version.
